In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Qiskit Runtime 로컬 테스트 모드

<span id="test-locally"></span>


<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하시기를 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
로컬 테스트 모드(`qiskit-ibm-runtime` v0.22.0 이상에서 사용 가능)를 사용하여 프로그램을 미세 조정하고 실제 양자 하드웨어에 전송하기 전에 테스트하세요. 로컬 테스트 모드로 프로그램을 검증한 후에는 QPU에서 실행하기 위해 Backend 이름만 변경하면 됩니다.

로컬 테스트 모드를 사용하려면 ``qiskit_ibm_runtime.fake_provider``의 페이크 Backend 중 하나를 지정하거나, Qiskit Runtime 기본 요소 또는 세션을 인스턴스화할 때 Qiskit Aer Backend를 지정하세요.

- **페이크 Backend**: ``qiskit_ibm_runtime.fake_provider``의 [페이크 Backend](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider)는 QPU 스냅샷을 사용하여 IBM&reg; QPU의 동작을 모방합니다. QPU 스냅샷에는 결합 맵, 기저 Gate, Qubit 속성 등 QPU에 대한 중요한 정보가 포함되어 있으며, 이는 Transpiler 테스트 및 QPU의 노이즈 시뮬레이션 수행에 유용합니다. 스냅샷의 노이즈 모델은 시뮬레이션 중에 자동으로 적용됩니다.
- **Aer 시뮬레이터**: [Qiskit Aer](/guides/simulate-with-qiskit-aer)의 시뮬레이터는 더 큰 Circuit과 [사용자 정의 노이즈 모델](/guides/build-noise-models)을 처리할 수 있는 고성능 시뮬레이션을 제공합니다. 로컬 테스트 모드에서 `AerSimulator`를 사용하면 다양한 시뮬레이션 방법 옵션을 사용할 수 있습니다. 많은 수의 Qubit을 가진 Clifford Circuit을 효율적으로 시뮬레이션하는 방법을 보여주는 [Clifford 시뮬레이션 모드 예제](#clifford-sim)를 참고하세요.

    <details>
    <summary>**Qiskit Aer에서 사용 가능한 시뮬레이션 방법 목록**</summary>

    자세한 내용은 [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) 문서를 참고하세요.

    * ``"automatic"``: 기본 시뮬레이션 방법입니다. Circuit과 노이즈 모델을 기반으로 시뮬레이션 방법을 자동으로 선택합니다.

    * ``"statevector"``: 모든 측정이 Circuit 끝에 있는 *이상적인* Circuit에서 측정 결과를 샘플링할 수 있는 밀집 상태 벡터 시뮬레이션입니다. 노이즈 시뮬레이션의 경우, 각 샷은 노이즈 모델에서 무작위로 샘플링된 노이즈 Circuit을 샘플링합니다.

    * ``"density_matrix"``: 모든 측정이 Circuit 끝에 있는 *노이즈* Circuit에서 측정 결과를 샘플링할 수 있는 밀도 행렬 시뮬레이션입니다.

    * ``"stabilizer"``: 노이즈 모델의 모든 오류가 Clifford 오류인 경우 노이즈 Clifford Circuit을 시뮬레이션할 수 있는 효율적인 Clifford 안정화 상태 시뮬레이터입니다.

    * ``"extended_stabilizer"``: 상태를 순위별 안정화 상태로 분해하는 것을 기반으로 하는 Clifford + T Circuit의 근사 시뮬레이터입니다. 항의 수는 비-Clifford(T) Gate의 수에 따라 증가합니다.

    * ``"matrix_product_state"``: 상태에 대해 행렬 곱 상태(MPS) 표현을 사용하는 텐서 네트워크 상태 벡터 시뮬레이터입니다. 시뮬레이터 옵션에 따라 MPS 결합 차원을 자르거나 자르지 않고 수행할 수 있습니다. 기본 동작은 잘라내지 않는 것입니다.

    * ``"unitary"``: 이상적인 Circuit의 밀집 단일 행렬 시뮬레이션입니다. 초기 양자 상태의 진화가 아닌 Circuit 자체의 단일 행렬을 시뮬레이션합니다. 이 방법은 Gate만 시뮬레이션할 수 있으며, 측정, 리셋 또는 노이즈를 지원하지 않습니다.

    * ``"superop"``: 이상적이거나 노이즈가 있는 Circuit의 밀집 슈퍼연산자 행렬 시뮬레이션입니다. 초기 양자 상태의 진화가 아닌 Circuit 자체의 슈퍼연산자 행렬을 시뮬레이션합니다. 이 방법은 이상적이고 노이즈가 있는 Gate와 리셋을 시뮬레이션할 수 있지만, 측정을 지원하지 않습니다.

    * ``"tensor_network"``: 상태 벡터와 밀도 행렬 모두를 지원하는 텐서 네트워크 기반 시뮬레이션입니다. 현재 GPU에서만 사용 가능하며 cuQuantum `cuTensorNet` API를 사용하여 가속화됩니다.
    </details>

> **Note:** - 로컬 테스트 모드에서 모든 Qiskit Runtime 옵션을 지정할 수 있습니다. 그러나 로컬 시뮬레이터에서 실행할 때 샷을 제외한 모든 옵션은 무시됩니다.
> - 페이크 Backend 또는 Aer 시뮬레이터를 사용하기 전에 `pip install qiskit-aer`를 실행하여 Qiskit Aer를 설치하는 것을 권장합니다. 페이크 Backend는 성능 향상을 위해 가능한 경우 내부적으로 Aer 시뮬레이터를 사용합니다.


## 페이크 Backend 예제

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## AerSimulator 예제
세션을 사용하는 예제, 노이즈 없음:

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

노이즈로 시뮬레이션하려면 QPU(양자 하드웨어)를 지정하고 Aer에 제출하세요. Aer는 해당 QPU의 보정 데이터를 기반으로 노이즈 모델을 구축하고, 해당 모델로 Aer Backend를 인스턴스화합니다. 원하는 경우 [노이즈 모델을 직접 구축](/guides/build-noise-models)할 수도 있습니다.

> **Caution:** QPU는 다양한 종류의 노이즈에 영향을 받을 수 있습니다. 여기서 사용되는 Qiskit Aer 노이즈 모델은 그 중 일부만 시뮬레이션하므로 실제 QPU의 노이즈보다 덜 심각할 가능성이 높습니다.
> 
> QPU에서 노이즈 모델을 초기화할 때 포함되는 오류에 대한 자세한 내용은 Aer [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend) API 참조를 확인하세요.

노이즈가 있는 예제:

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### Clifford 시뮬레이션
Clifford Circuit은 검증 가능한 결과로 효율적으로 시뮬레이션할 수 있으므로 Clifford 시뮬레이션은 매우 유용한 도구입니다. 심층적인 예제는 [Qiskit Aer 기본 요소를 사용한 안정화 Circuit의 효율적인 시뮬레이션](/guides/simulate-stabilizer-circuits)을 참고하세요.

예제:

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## 다음 단계
> **Tip:** - 자세한 [기본 요소 예제](/guides/primitives-examples)를 검토하세요.
>     - [V2 기본 요소로 마이그레이션](/guides/v2-primitives)을 읽어보세요.
>     - IBM Quantum Learning의 [비용 함수 레슨](/learning/courses/variational-algorithm-design/cost-functions)을 통해 기본 요소를 연습하세요.
>     - [Transpile](/guides/transpile) 섹션에서 로컬로 트랜스파일하는 방법을 알아보세요.
>     - [Transpiler 설정 비교](/guides/circuit-transpilation-settings#compare-transpiler-settings) 튜토리얼을 시도해 보세요.